# modist in Jupyter

Interactive distribution widgets work in any anywidget-compatible environment.
No marimo required — just `display()` the widget and read `.params`.

`w.params` is a plain dict of the synced traits. Drag a widget, then re-run
a cell that reads `.params` to see the values update.

In [ ]:
import modist as md
from IPython.display import display

## Normal

Drag the **● mean line** to reposition, or either **■ ±1σ square** to reshape.

In [ ]:
normal = md.Normal(mu=0, sigma=1)
display(normal)

In [ ]:
normal.params

## Beta

Domain is fixed to **[0, 1]**. Drag the **● mean line** to translate, or the
**■ quartile squares** to concentrate / spread.

In [ ]:
beta = md.Beta(alpha=2, beta=5)
display(beta)

In [ ]:
beta.params

## Gamma

Left edge pinned at **0**. Drag the **● mean line** to translate, or the
**■ quartile squares** to reshape.

In [ ]:
gamma = md.Gamma(alpha=2, beta=2)
display(gamma)

In [ ]:
gamma.params

## Student-t

Unbounded support. Drag the **● mean line** to shift (`mu`). The **■ q75
square** edits spread (`sigma`, horizontal) or tail weight (`nu`, vertical).

In [ ]:
student_t = md.StudentT(mu=0, sigma=1, nu=5)
display(student_t)

In [ ]:
student_t.params

## Using `.scipy` for live stats

Each widget has a lazy `.scipy` property that returns a frozen scipy
distribution with the current params. Drag the widget, then re-run the cell
to see the numbers update.

In [ ]:
from scipy import stats
import pandas as pd

n = normal.scipy  # frozen scipy.stats.norm(mu, sigma)
pd.DataFrame({
    "metric": ["mean", "std", "median", "P(±1σ)", "95% CI"],
    "value": [
        round(n.mean(), 4),
        round(n.std(), 4),
        round(n.ppf(0.5), 4),
        round(n.cdf(n.mean() + n.std()) - n.cdf(n.mean() - n.std()), 4),
        f"[{n.ppf(0.025):.2f}, {n.ppf(0.975):.2f}]",
    ],
})

## Direct scipy splat

Each widget has a lazy `.scipy` property that returns a frozen `scipy.stats`
distribution with the current params — handling the per-family parametrization
(`mu`/`sigma` -> `loc`/`scale`, Gamma shape/rate -> shape/scale, etc.):

In [ ]:
n = normal.scipy   # stats.norm(loc=mu, scale=sigma)
b = beta.scipy     # stats.beta(a=alpha, b=beta)
g = gamma.scipy    # stats.gamma(a=alpha, scale=1/beta)

print(n)
print(b)
print(g)

## Combining widgets

Use multiple widgets together — e.g. a simple Bayesian prior sensitivity
check with a Beta prior on a coin's bias.

In [ ]:
prior_uniform = md.Beta(alpha=1, beta=1)
prior_skeptical = md.Beta(alpha=2, beta=5)

display(prior_uniform)
display(prior_skeptical)

In [ ]:
d_uniform = prior_uniform.scipy
d_skeptical = prior_skeptical.scipy
print(f"uniform:   Beta(1, 1)     mean={d_uniform.mean():.3f}")
print(f"skeptical: Beta(2, 5)     mean={d_skeptical.mean():.3f}")